In [0]:
from pyspark.sql import functions as F

df = spark.table("internet_fijo_elt.silver.conexiones_internet_fijo")

# Traer las dimensiones para hacer join y obtener los IDs
dim_periodo   = spark.table("internet_fijo_elt.gold.dim_periodo")
dim_empresa   = spark.table("internet_fijo_elt.gold.dim_empresa")
dim_segmento  = spark.table("internet_fijo_elt.gold.dim_segmento")
dim_tecnologia = spark.table("internet_fijo_elt.gold.dim_tecnologia")
dim_ubicacion = spark.table("internet_fijo_elt.gold.dim_ubicacion")

fact_conexiones = (
    df
    .join(dim_periodo, df.periodo == dim_periodo.periodo, "left")
    .join(dim_empresa, df.empresa == dim_empresa.nombre_empresa, "left")
    .join(dim_tecnologia, df.tecnologia == dim_tecnologia.tecnologia, "left")
    .join(dim_ubicacion, df.provincia == dim_ubicacion.provincia, "left")
    .join(dim_segmento, df.segmento == dim_segmento.segmento, "left")  
    .select(
        dim_periodo["fecha_id"],
        dim_empresa["id_empresa"],
        dim_tecnologia["id_tecnologia"],
        dim_ubicacion["id_ubicacion"],
        dim_segmento["id_segmento"],
        df["conexiones"]
    )
)

fact_conexiones.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("internet_fijo_elt.gold.fact_conexiones")

In [0]:
spark.sql("SHOW TABLES IN internet_fijo_elt.gold").show()

+--------+---------------+-----------+
|database|      tableName|isTemporary|
+--------+---------------+-----------+
|    gold|    dim_empresa|      false|
|    gold|    dim_periodo|      false|
|    gold|   dim_segmento|      false|
|    gold| dim_tecnologia|      false|
|    gold|  dim_ubicacion|      false|
|    gold|fact_conexiones|      false|
+--------+---------------+-----------+

